In [ ]:
# Jalankan kalau belum pernah install

SyntaxError: invalid syntax (3979995109.py, line 2)

In [3]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog

from skimage import measure
from skimage import morphology

In [4]:
def labeled_user_image(image,k = 0):

    imgray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    (thresh, im_bw) = cv2.threshold(imgray, 90, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    #im_bw= cv2.adaptiveThreshold(imgray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY, 11, 2)
    im_bw = np.array(255-im_bw, dtype=bool)
    cleaned = morphology.remove_small_objects(im_bw, min_size=20, connectivity=2)
    cleaned = np.array(cleaned, dtype=int)
    cleaned = 255+cleaned
    label, n = measure.label(cleaned, background=255, return_num=True, connectivity=2)
    print("numbers of numpers on image : ",n)
    x = []
    y = []
    numbers = []
    ph=[]
    rect=[]
    for i in range(1, n + 1):
        for r in range(label.shape[0]):
            for c in range(label.shape[1]):
                if label[r, c] == i:
                    x.append(r)
                    y.append(c)

        digit = im_bw[min(x): max(x), min(y): max(y)]

        rect.append([(min(y), min(x)), (max(y) - min(y)), (max(x) - min(x))])
        padd_y = 0
        padding = np.zeros([digit.shape[0]+padd_y, digit.shape[1] + k], dtype='float64')
        padding[padd_y//2:padding.shape[0]-padd_y//2, k//2:padding.shape[1] - k//2] = digit
        ph.append(padding)

        re_digit= cv2.resize(np.array(padding,dtype='float64'), (28, 28), interpolation=cv2.INTER_AREA)
        re_digit = cv2.dilate(re_digit, (3, 3))
        # Calculate the HOG features
        roi_hog_fd = hog(re_digit, orientations=9, pixels_per_cell=(14, 14), cells_per_block=(1, 1), visualize=False)

        numbers.append( np.array([roi_hog_fd], 'float64'))
        x = []
        y = []

    return numbers,ph,rect

In [6]:
# Cara penggunaan
image_path = '73.png'
image = cv2.imread(image_path)
numbers, ph, rect = labeled_user_image(image)

# rect berupa list titik koordinat objek dan lebar serta tinggi
# gunakan value dari rect untuk melakukan prediction

numbers of numpers on image :  2


In [10]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Load the pre-trained MNIST model (replace with your model path if necessary)
model = load_model("mnist_model.h5")

def preprocess_image(image):
    """Convert input image to grayscale and apply thresholding."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY_INV)
    return thresh

def extract_digits(image):
    """Detect digits in the image and extract ROIs."""
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rois = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if h > 10 and w > 10:  # Filter out noise by size
            roi = image[y:y+h, x:x+w]
            rois.append((x, roi))
    # Sort ROIs by their x-coordinates (left to right)
    rois = sorted(rois, key=lambda r: r[0])
    return [r[1] for r in rois]

def recognize_digits(rois):
    """Predict digits from extracted ROIs using the trained model."""
    digits = []
    for roi in rois:
        roi_resized = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
        roi_resized = roi_resized.astype("float32") / 255.0  # Normalize to [0, 1]
        roi_resized = img_to_array(roi_resized)
        roi_resized = np.expand_dims(roi_resized, axis=0)  # Add batch dimension
        prediction = model.predict(roi_resized)
        digit = np.argmax(prediction)
        digits.append(digit)
    return digits

def main(image_path):
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print("Error loading image!")
        return

    # Preprocess the image
    preprocessed = preprocess_image(image)

    # Extract digits from the image
    rois = extract_digits(preprocessed)

    # Recognize digits in each ROI
    digits = recognize_digits(rois)

    return digits

# Example usage
if __name__ == "__main__":
    image_path = "73.png"
    detected_digits = main(image_path)
    print("Detected digits:", detected_digits)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Detected digits: [1, 7]


In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Step 1: Load and preprocess the MNIST dataset
def load_data():
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    # Normalize the pixel values to the range [0, 1]
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # Reshape data to include a channel dimension (28x28x1)
    x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
    x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

    # One-hot encode the labels
    y_train = to_categorical(y_train, 10)
    y_test = to_categorical(y_test, 10)

    return (x_train, y_train), (x_test, y_test)

# Step 2: Define the CNN model
def create_model():
    model = Sequential([
        # Convolutional layer with 32 filters, 3x3 kernel size, ReLU activation
        Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),  # Max-pooling with 2x2 pool size

        # Second convolutional layer
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        Flatten(),  # Flatten the feature maps into a 1D feature vector
        Dense(128, activation='relu'),  # Fully connected layer
        Dropout(0.5),  # Dropout for regularization
        Dense(10, activation='softmax')  # Output layer for classification
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Step 3: Train and save the model
def train_and_save_model():
    (x_train, y_train), (x_test, y_test) = load_data()

    model = create_model()
    
    # Train the model
    model.fit(x_train, y_train, epochs=100, batch_size=64, validation_split=0.2)

    # Evaluate the model on the test data
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
    print(f"Test accuracy: {test_acc}")

    # Save the trained model
    model.save("mnist_model.h5")
    print("Model saved as mnist_model.h5")

# Run the training process
if __name__ == "__main__":
    train_and_save_model()


Epoch 1/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8210 - loss: 0.5622 - val_accuracy: 0.9800 - val_loss: 0.0684
Epoch 2/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9707 - loss: 0.0946 - val_accuracy: 0.9854 - val_loss: 0.0501
Epoch 3/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9780 - loss: 0.0746 - val_accuracy: 0.9876 - val_loss: 0.0416
Epoch 4/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9838 - loss: 0.0530 - val_accuracy: 0.9877 - val_loss: 0.0396
Epoch 5/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9846 - loss: 0.0492 - val_accuracy: 0.9883 - val_loss: 0.0389
Epoch 6/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9866 - loss: 0.0429 - val_accuracy: 0.9896 - val_loss: 0.0380
Epoch 7/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9896 - loss: 0.0320 - val_accuracy: 0.9899 - val_loss: 0.0385
Epoch 8/100
750/750 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9905 - loss: 0.0303

Test accuracy: 0.9936000108718872
Model saved as mnist_model.h5


In [16]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Load the pre-trained MNIST model
model = load_model("mnist_model.h5")

def preprocess_image(image):
    """
    Convert input image to grayscale and apply thresholding to prepare for contour detection.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    # Apply adaptive thresholding for robust binarization
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
    )
    return thresh

def extract_digits(image):
    """
    Detect digits in the image and extract ROIs.
    """
    # Find contours
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rois = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # Filter by size to exclude noise
        if h > 10 and w > 10:  
            roi = image[y:y + h, x:x + w]
            rois.append((x, roi))  # Store the x-coordinate for sorting
    # Sort ROIs by their x-coordinate (left-to-right order)
    rois = sorted(rois, key=lambda r: r[0])
    return [r[1] for r in rois]

def recognize_digits(rois):
    """
    Predict digits from extracted ROIs using the trained MNIST model.
    """
    digits = []
    for roi in rois:
        # Resize ROI to 28x28 pixels (MNIST model input size)
        roi_resized = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
        roi_resized = roi_resized.astype("float32") / 255.0  # Normalize to [0, 1]
        roi_resized = np.expand_dims(img_to_array(roi_resized), axis=0)  # Add batch dimension
        prediction = model.predict(roi_resized)  # Get model predictions
        digit = np.argmax(prediction)  # Get the class with the highest probability
        digits.append(digit)
    return digits

def main(image_path):
    """
    Main function to process the input image and return detected digits.
    """
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print("Error: Image could not be loaded!")
        return []

    # Preprocess the image
    preprocessed = preprocess_image(image)

    # Extract digits from the image
    rois = extract_digits(preprocessed)

    # Recognize digits in each ROI
    digits = recognize_digits(rois)

    return digits

# Example usage
if __name__ == "__main__":
    image_path = "73.png"  # Replace with the path to your image
    detected_digits = main(image_path)
    print("Detected digits:", detected_digits)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Detected digits: [1, 5]


In [17]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# Load the pre-trained MNIST model
model = load_model("mnist_model.h5")  # Make sure this is in the same directory or provide the correct path

def preprocess_image(image):
    """
    Preprocess the input image: convert to grayscale, apply adaptive thresholding.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    # Apply adaptive thresholding for robust binarization
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
    )
    return thresh

def extract_digits(thresh_image):
    """
    Detect digits in the thresholded image and extract their ROIs.
    """
    # Find contours
    contours, _ = cv2.findContours(thresh_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rois = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # Filter by size to exclude noise
        if h > 10 and w > 10:
            roi = thresh_image[y:y + h, x:x + w]
            rois.append((x, roi))  # Store the x-coordinate for sorting
    # Sort ROIs by their x-coordinate (left-to-right order)
    rois = sorted(rois, key=lambda r: r[0])
    return [r[1] for r in rois]

def recognize_digits(rois):
    """
    Predict digits from extracted ROIs using the trained MNIST model.
    """
    digits = []
    for roi in rois:
        # Resize ROI to 28x28 pixels (MNIST model input size)
        roi_resized = cv2.resize(roi, (28, 28), interpolation=cv2.INTER_AREA)
        roi_resized = roi_resized.astype("float32") / 255.0  # Normalize to [0, 1]
        roi_resized = np.expand_dims(img_to_array(roi_resized), axis=0)  # Add batch dimension
        prediction = model.predict(roi_resized)  # Get model predictions
        digit = np.argmax(prediction)  # Get the class with the highest probability
        digits.append(digit)
    return digits

def main(image_path):
    """
    Main function to process the input image and return detected digits.
    """
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print("Error: Image could not be loaded!")
        return []

    # Preprocess the image
    preprocessed = preprocess_image(image)

    # Extract digits from the image
    rois = extract_digits(preprocessed)

    # Recognize digits in each ROI
    digits = recognize_digits(rois)

    return digits

# Example usage
if __name__ == "__main__":
    image_path = "73.png"  # Replace with the path to your image
    detected_digits = main(image_path)
    print("Detected digits:", detected_digits)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Detected digits: [1, 5]
